In [1]:
import numpy as np
import polars as pl

import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go

import requests as rq
from datetime import datetime
import os
os.environ['KERAS_BACKEND'] = 'torch'

In [2]:
import keras

In [3]:
def get_belpex_data(start_date: str, end_date: str, duration_choice: int) -> dict:
    url = 'https://api.fibonaki.dev/api/get_elec_epex_spot_data'

    params = {
        'start_date': start_date, #yyyy-mm-dd
        'end_data': end_date,
        'duration_choice': duration_choice
    }

    results = rq.get(url, params=params).json()
    return results

def get_dim_date(start: datetime, end: datetime) -> pl.DataFrame:
    dim_date = pl.DataFrame(
        pl.datetime_range(
            start = start, 
            end = end,
            interval = '1h',
            eager = True
        )
    ).rename({'literal': 'date'})
    
    return dim_date

In [4]:
# start_date = datetime(2025, 5, 15)
# end_date = datetime(2023, 1, 1)
# delta = (datetime(2025, 5, 15)-datetime(2023, 1, 1)).days

# data = get_belpex_data('2023-01-01', '2025-05-15', duration_choice=delta)
# data['elec_epex_spot_data'].pop('last_week_belpex_data')

# pl.from_dict(data['elec_epex_spot_data']).with_columns(
#     belpex_labels = pl.col('belpex_labels').str.to_datetime(),
#     belpex_values = pl.col('belpex_values').cast(pl.Float64)
# ).write_parquet('data/belpex.parquet')

In [5]:
data = pl.read_parquet('data/belpex.parquet').rename({'belpex_labels': 'date'}).sort(pl.col('date'))
print(data.shape)
data.head(3)

(20275, 2)


date,belpex_values
datetime[μs],f64
2023-01-11 00:00:00,5.64
2023-01-11 01:00:00,0.31
2023-01-11 02:00:00,0.27


# mind the gap

On perd des heures lors des changements d'heures.

C'est plus simple des les combler

<br>

In [6]:
data.select(pl.col('date').min().alias('min'), pl.col('date').max().alias('max'))

start_date = data.select(pl.col('date').min()).get_column('date')[0]
end_date = data.select(pl.col('date').max()).get_column('date')[0]

dim_date = get_dim_date(start_date, end_date)
data = dim_date.join(
    data, on='date', how='left'
).fill_null(strategy='forward')

data

date,belpex_values
datetime[μs],f64
2023-01-11 00:00:00,5.64
2023-01-11 01:00:00,0.31
2023-01-11 02:00:00,0.27
2023-01-11 03:00:00,-0.08
2023-01-11 04:00:00,-1.04
…,…
2025-05-24 19:00:00,72.83
2025-05-24 20:00:00,92.58
2025-05-24 21:00:00,95.09


# split dataset

<br>

In [7]:
def split_dataset(window_size: int) -> pl.DataFrame:

    data = pl.read_parquet('data/belpex.parquet').rename({'belpex_labels': 'date'}).sort(pl.col('date'))

    data.select(pl.col('date').min().alias('min'), pl.col('date').max().alias('max'))

    start_date = data.select(pl.col('date').min()).get_column('date')[0]
    end_date = data.select(pl.col('date').max()).get_column('date')[0]

    dim_date = get_dim_date(start_date, end_date)
    data = dim_date.join(
        data, on='date', how='left'
    ).fill_null(strategy='forward')

    cutoff_train_test = int(data.shape[0]*0.90)/window_size
    cutoff_train_val = int(cutoff_train_test*0.80)
    
    return data.with_row_index(offset=0)\
        .with_columns(sequence=pl.col('index') // window_size)\
        .with_columns(
        dataset=pl.when(pl.col('sequence') <= cutoff_train_val).then(pl.lit('train'))
        .when(pl.col('sequence') <= cutoff_train_test).then(pl.lit('val'))
        .otherwise(pl.lit('test'))
        )

window_size = 4
data = split_dataset(window_size)
data

index,date,belpex_values,sequence,dataset
u32,datetime[μs],f64,u32,str
0,2023-01-11 00:00:00,5.64,0,"""train"""
1,2023-01-11 01:00:00,0.31,0,"""train"""
2,2023-01-11 02:00:00,0.27,0,"""train"""
3,2023-01-11 03:00:00,-0.08,0,"""train"""
4,2023-01-11 04:00:00,-1.04,1,"""train"""
…,…,…,…,…
20755,2025-05-24 19:00:00,72.83,5188,"""test"""
20756,2025-05-24 20:00:00,92.58,5189,"""test"""
20757,2025-05-24 21:00:00,95.09,5189,"""test"""


In [8]:
data_windowed = data.select(pl.col('date', 'belpex_values', 'dataset'))\
    .group_by_dynamic('date', every='1h', period='48h')\
    .agg(
        pl.col('belpex_values').mean(),
        pl.col('dataset').max()
    )

m_avg = data.select(pl.col('date', 'belpex_values', 'dataset'))\
    .group_by_dynamic('date', every='1h', period=f'1w')\
    .agg(
        pl.col('belpex_values').median(),
        pl.col('dataset').max()
    )

# plt.figure(figsize=(20, 8))
# sns.lineplot(data, x='date', y='belpex_values', hue='dataset')
# sns.lineplot(m_avg, x='date', y='belpex_values', c='red')
# plt.axhline(0, c='r')
# plt.show()

# plt.figure(figsize=(20, 8))
# sns.lineplot(data_windowed, x='date', y='belpex_values', hue='dataset')
# plt.show()

In [9]:
fig1 = go.Figure()
for ds in data['dataset'].unique():
    subset = data.filter(pl.col('dataset') == ds)
    fig1.add_trace(go.Scatter(
        x=subset['date'],
        y=subset['belpex_values'],
        mode='lines',
        name=f'{ds} actual'
    ))
fig1.add_trace(go.Scatter(
    x=m_avg['date'],
    y=m_avg['belpex_values'],
    mode='lines',
    name='median avg',
    line=dict(color='red')
))
fig1.update_layout(
    title='Belpex Values and Median Average',
    xaxis_title='Date',
    yaxis_title='Belpex Value',
    width=1200,
    height=500,
    xaxis=dict(
        rangeslider=dict(visible=True),
        type='date'
    )
)
fig1.show()

# Second graph: windowed mean
fig2 = go.Figure()
for ds in data_windowed['dataset'].unique():
    subset = data_windowed.filter(pl.col('dataset') == ds)
    fig2.add_trace(go.Scatter(
        x=subset['date'],
        y=subset['belpex_values'],
        mode='lines',
        name=f'{ds} windowed mean'
    ))
fig2.update_layout(
    title='Windowed Mean Belpex Values',
    xaxis_title='Date',
    yaxis_title='Belpex Value',
    width=1200,
    height=500,
    xaxis=dict(
        rangeslider=dict(visible=True),
        type='date'
    )
)
fig2.show()

# Non overlapping windows

<br>

## Split xy

<br>

In [10]:
def build_non_overlapping_windows(data: pl.DataFrame, dataset_name: str) -> pl.DataFrame:
    return data.filter(pl.col('dataset') == dataset_name)\
        .group_by('sequence', maintain_order=True)\
        .agg(
            pl.col('date'),
            pl.col('belpex_values'),
        )\
        .with_columns(
            target=pl.col('belpex_values').list.first().shift(-1),
            target_date=pl.col('date').list.first().shift(-1)
        )\
        .slice(0, -1)

def split_xy(data: pl.DataFrame) -> tuple[np.array, np.array]:
    x = np.array(data.get_column('belpex_values').to_list())
    x = x[..., np.newaxis]
    y = np.array(data.get_column('target').to_list())
    return x, y

In [11]:
window_size = 4
data = split_dataset(window_size)

datasets = {}
for split in ['train', 'val', 'test']:
    datasets[split] = build_non_overlapping_windows(data, split)

display(datasets['train'].head(3))

x_train, y_train = split_xy(datasets.get('train'))
x_val, y_val = split_xy(datasets.get('val'))
x_test, y_test = split_xy(datasets.get('test'))
print(x_train.shape)

sequence,date,belpex_values,target,target_date
u32,list[datetime[μs]],list[f64],f64,datetime[μs]
0,"[2023-01-11 00:00:00, 2023-01-11 01:00:00, … 2023-01-11 03:00:00]","[5.64, 0.31, … -0.08]",-1.04,2023-01-11 04:00:00
1,"[2023-01-11 04:00:00, 2023-01-11 05:00:00, … 2023-01-11 07:00:00]","[-1.04, 2.89, … 94.29]",141.99,2023-01-11 08:00:00
2,"[2023-01-11 08:00:00, 2023-01-11 09:00:00, … 2023-01-11 11:00:00]","[141.99, 145.72, … 131.12]",119.16,2023-01-11 12:00:00


(3736, 4, 1)


In [12]:
from keras import Model, Input
from keras.layers import Dense, Conv1D, MaxPool1D, Flatten, SimpleRNN, Concatenate
from keras.optimizers import Adam
from keras.losses import mean_squared_error

from keras.utils import plot_model

## CNN

<br>

In [13]:
def get_cnn(window_size: int) -> Model:
    inputs = Input((window_size, 1))

    x = Conv1D(64, 3, padding='same', activation='relu')(inputs)
    x = Conv1D(64, 3, padding='same', activation='relu')(x)
    x = MaxPool1D(2)(x)

    x = Conv1D(128, 3, padding='same', activation='relu')(x)
    x = Conv1D(128, 3, padding='same', activation='relu')(x)
    x = MaxPool1D(2)(x)

    x = Flatten()(x)

    x = Dense(128, activation='relu')(x)
    outputs = Dense(1, activation='linear')(x)

    cnn_model = Model(inputs, outputs)

    cnn_model.compile(
        optimizer=Adam()
        , loss=mean_squared_error
        , metrics=['MAE']
    )

    return cnn_model

cnn_model = get_cnn(window_size)
cnn_history = cnn_model.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=25, batch_size=512)

Epoch 1/25
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - MAE: 77.8298 - loss: 7949.2793 - val_MAE: 26.6804 - val_loss: 1471.1847
Epoch 2/25
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - MAE: 29.3109 - loss: 1406.6609 - val_MAE: 20.2011 - val_loss: 967.6721
Epoch 3/25
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - MAE: 21.4422 - loss: 864.5439 - val_MAE: 17.6035 - val_loss: 644.5359
Epoch 4/25
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - MAE: 17.1170 - loss: 485.3860 - val_MAE: 13.9238 - val_loss: 419.5108
Epoch 5/25
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - MAE: 10.5626 - loss: 226.6420 - val_MAE: 11.7307 - val_loss: 252.6909
Epoch 6/25
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - MAE: 9.9209 - loss: 196.5207 - val_MAE: 9.9391 - val_loss: 221.5521
Epoch 7/25
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - MAE: 8.7524 - loss: 167.0974 - val_MAE: 9.0139 - val_loss: 214.6136
Epoch 8/25
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - MAE: 8.2510 - loss: 153.4604 - val_MAE: 8.7234 - val_loss: 193.3453
Epoch 9/25
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 9

## RNN

<br>

In [14]:
def get_rnn(window_size: int) -> Model:
    inputs = Input((window_size, 1))
    x = SimpleRNN(128, activation='relu', return_sequences=True)(inputs)
    x = SimpleRNN(64, activation='relu')(x)
    x = Dense(64, activation='relu')(x)
    outputs = Dense(1, activation='linear')(x)

    rnn_model = Model(inputs, outputs)
    rnn_model.compile(
        optimizer=Adam(),
        loss=mean_squared_error,
        metrics=['MAE']
    )

    return rnn_model

rnn_model = get_rnn(window_size)
rrn_history = rnn_model.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=25, batch_size=512)

Epoch 1/25
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - MAE: 86.0938 - loss: 9699.3750 - val_MAE: 54.3497 - val_loss: 4149.0244
Epoch 2/25
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - MAE: 34.6341 - loss: 1955.9688 - val_MAE: 33.2566 - val_loss: 1854.4166
Epoch 3/25
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - MAE: 25.0756 - loss: 1012.4154 - val_MAE: 23.0719 - val_loss: 1080.5787
Epoch 4/25
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - MAE: 18.3359 - loss: 656.1319 - val_MAE: 16.2782 - val_loss: 662.4550
Epoch 5/25
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - MAE: 14.4633 - loss: 430.6091 - val_MAE: 15.3916 - val_loss: 616.8390
Epoch 6/25
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - MAE: 12.7719 - loss: 341.6250 - val_MAE: 13.3318 - val_loss: 454.5802
Epoch 7/25
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - MAE: 10.8206 - loss: 256.9930 - val_MAE: 11.3261 - val_loss: 322.6093
Epoch 8/25
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - MAE: 9.1847 - loss: 201.0986 - val_MAE: 9.8040 - val_loss: 247.2249
Epoch 9/25
8/8 ━━━━━━━━━━━━━

## Plot result

<br>

In [15]:
y_pred_cnn = cnn_model.predict(x_test)
y_pred_rnn = rnn_model.predict(x_test)


17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


In [16]:
plot_testset = datasets.get('test').with_columns(
    pl.Series('y_pred_cnn', y_pred_cnn.flatten()),
    pl.Series('y_pred_rnn', y_pred_rnn.flatten()),
    ).slice(0, -1).select(pl.exclude('belpex_values', 'date'))
plot_testset.head(3)

sequence,target,target_date,y_pred_cnn,y_pred_rnn
u32,f64,datetime[μs],f32,f32
4672,136.88,2025-02-27 20:00:00,156.273224,153.45282
4673,115.25,2025-02-28 00:00:00,109.158234,104.055428
4674,109.8,2025-02-28 04:00:00,106.779297,103.577225


In [17]:
# plt.figure(figsize=(20, 8))
# sns.lineplot(data.filter(pl.col('dataset') == 'test'), x='date', y='belpex_values', label='actual')
# sns.lineplot(plot_testset, x='target_date', y='y_pred_cnn', label='cnn_predicted')
# sns.lineplot(plot_testset, x='target_date', y='y_pred_rnn', label='rnn_predicted')
# plt.legend()
# plt.show()

In [18]:
fig = go.Figure()

# Actual values
fig.add_trace(go.Scatter(
    x=data.filter(pl.col('dataset') == 'test')['date'],
    y=data.filter(pl.col('dataset') == 'test')['belpex_values'],
    mode='lines',
    name='actual'
))

fig.add_trace(go.Scatter(
    x=plot_testset['target_date'],
    y=plot_testset['y_pred_cnn'],
    mode='lines',
    name='cnn_predicted'
))

fig.add_trace(go.Scatter(
    x=plot_testset['target_date'],
    y=plot_testset['y_pred_rnn'],
    mode='lines',
    name='rnn_predicted'
))

fig.update_layout(
    title='Test Set Predictions',
    xaxis_title='Date',
    yaxis_title='Belpex Value',
    xaxis=dict(
        rangeslider=dict(visible=True),
        type='date'
    ),
    width=1200,
    height=600
)

fig.show()

# With overlapping windows

<br>

In [19]:
def build_overlapping_windows(data: pl.DataFrame, window_size: int, dataset_name: str) -> pl.DataFrame:
    return data.filter(pl.col('dataset') == dataset_name)\
        .group_by_dynamic('date', every='1h', period=f'{window_size}h')\
        .agg(
            pl.col('belpex_values'),
            pl.len(),
            )\
        .with_columns(
            target = pl.col('belpex_values').list.last().shift(-1),
            target_date = pl.col('date').shift(-window_size),
            )\
        .filter(pl.col('len') == window_size)\
        .slice(0, -1)

In [20]:
window_size = 12
data = split_dataset(window_size)

datasets = {}
for split in ['train', 'val', 'test']:
    datasets[split] = build_overlapping_windows(data, window_size, split)

x_train, y_train = split_xy(datasets.get('train'))
x_val, y_val = split_xy(datasets.get('val'))
x_test, y_test = split_xy(datasets.get('test'))
print(x_train.shape)

(14940, 12, 1)


In [21]:
cnn_model = get_cnn(window_size)
cnn_history = cnn_model.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=25, batch_size=512)

rnn_model = get_rnn(window_size)
rnn_history = rnn_model.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=25, batch_size=512)

Epoch 1/25
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - MAE: 40.5543 - loss: 2857.6431 - val_MAE: 16.9911 - val_loss: 565.8223
Epoch 2/25
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - MAE: 12.1742 - loss: 278.7892 - val_MAE: 11.3539 - val_loss: 261.8434
Epoch 3/25
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - MAE: 9.7042 - loss: 187.9220 - val_MAE: 9.4199 - val_loss: 204.2826
Epoch 4/25
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - MAE: 8.7692 - loss: 160.4571 - val_MAE: 8.9168 - val_loss: 188.9579
Epoch 5/25
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - MAE: 8.4697 - loss: 155.2343 - val_MAE: 8.7060 - val_loss: 183.1621
Epoch 6/25
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - MAE: 8.3613 - loss: 151.1982 - val_MAE: 9.2893 - val_loss: 196.9761
Epoch 7/25
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - MAE: 8.1678 - loss: 141.7788 - val_MAE: 8.5621 - val_loss: 180.5711
Epoch 8/25
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - MAE: 8.1311 - loss: 143.3265 - val_MAE: 8.6605 - val_loss: 181.6199
Epoch 9/25
30/30 ━━━━━━━━━━━━━━━━━

In [22]:
y_pred_cnn = cnn_model.predict(x_test)
y_pred_rnn = rnn_model.predict(x_test)
plot_testset = datasets.get('test').with_columns(
    pl.Series('y_pred_cnn', y_pred_cnn.flatten()),
    pl.Series('y_pred_rnn', y_pred_rnn.flatten()),
    ).select(pl.exclude('belpex_values', 'date'))
plot_testset.head(3)

65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
65/65 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step


len,target,target_date,y_pred_cnn,y_pred_rnn
u32,f64,datetime[μs],f32,f32
12,100.42,2025-02-28 12:00:00,103.584389,102.587395
12,95.7,2025-02-28 13:00:00,94.34832,92.844505
12,98.07,2025-02-28 14:00:00,92.952194,89.624275


In [23]:
# plt.figure(figsize=(20, 8))
# sns.lineplot(plot_testset, x='target_date', y='target', label='actual')
# sns.lineplot(plot_testset, x='target_date', y='y_pred_cnn', label='cnn_predicted')
# sns.lineplot(plot_testset, x='target_date', y='y_pred_rnn', label='rnn_predicted')
# plt.legend()
# plt.show()

In [24]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=plot_testset['target_date'],
    y=plot_testset['target'],
    mode='lines',
    name='actual'
))

fig.add_trace(go.Scatter(
    x=plot_testset['target_date'],
    y=plot_testset['y_pred_cnn'],
    mode='lines',
    name='cnn_predicted'
))

fig.add_trace(go.Scatter(
    x=plot_testset['target_date'],
    y=plot_testset['y_pred_rnn'],
    mode='lines',
    name='rnn_predicted'
))

fig.update_layout(
    title='Test Set Predictions',
    xaxis_title='Date',
    yaxis_title='Belpex Value',
    xaxis=dict(
        rangeslider=dict(visible=True),
        type='date'
    ),
    width=1200,
    height=600
)

fig.show()

# With external datas

UTC timezone is not known for the epex spot.

Let's assume it is Europe/Brussels and let's add 1hour.

Not the point here to exactly sync the dates

<br>

In [25]:
window_size = 12
data = split_dataset(window_size)

weather_data = pl.read_csv('data/weather_data.csv', try_parse_dates=True)\
    .with_columns(date = pl.from_epoch(pl.col('dt')+3600))

weather_data = data.join(weather_data.select(pl.col('date', 'temp')), how='left', on='date').sort(pl.col('date'))

weather_data.head(3)

index,date,belpex_values,sequence,dataset,temp
u32,datetime[μs],f64,u32,str,f64
0,2023-01-11 00:00:00,5.64,0,"""train""",10.91
1,2023-01-11 01:00:00,0.31,0,"""train""",11.6
2,2023-01-11 02:00:00,0.27,0,"""train""",11.33


In [26]:
weather_data.select(pl.corr('belpex_values', 'temp'))

belpex_values
f64
-0.392268


In [27]:
def build_weather_windows(weather_data: pl.DataFrame, window_size: int, dataset_name: str) -> pl.DataFrame:
    return weather_data.filter(pl.col('dataset') == dataset_name)\
        .group_by_dynamic('date', every='1h', period=f'{window_size}h')\
        .agg(
            pl.col('belpex_values'),
            pl.col('temp'),
            pl.len()
            )\
        .with_columns(
            target=pl.col('belpex_values').list.last().shift(-1),
            target_date = pl.col('date').shift(-window_size)
            )\
        .filter(pl.col('len') == window_size)\
        .slice(0, -1)

def split_xy_two_heads(datasets):
    trainset = datasets['train']
    valset = datasets['val']
    testset = datasets['test']

    x_train_price = np.array(trainset.get_column('belpex_values').to_list())
    x_train_price = x_train_price[..., np.newaxis]
    x_train_temp = np.array(trainset.get_column('temp').to_list())
    x_train_temp = x_train_temp[..., np.newaxis]
    y_train = np.array(trainset.get_column('target').to_list())

    x_val_price = np.array(valset.get_column('belpex_values').to_list())
    x_val_price = x_val_price[..., np.newaxis]
    x_val_temp = np.array(valset.get_column('temp').to_list())
    x_val_temp = x_val_temp[..., np.newaxis]
    y_val = np.array(valset.get_column('target').to_list())

    x_test_price = np.array(testset.get_column('belpex_values').to_list())
    x_test_price = x_test_price[..., np.newaxis]
    x_test_temp = np.array(testset.get_column('temp').to_list())
    x_test_temp = x_test_temp[..., np.newaxis]
    y_test = np.array(testset.get_column('target').to_list())

    return (x_train_price, x_train_temp, y_train,
            x_val_price, x_val_temp, y_val,
            x_test_price, x_test_temp, y_test)

In [28]:
datasets = {}
for split in ['train', 'val', 'test']:
    datasets[split] = build_weather_windows(weather_data, window_size, split)

x_train_price, x_train_temp, y_train, \
x_val_price, x_val_temp, y_val, \
x_test_price, x_test_temp, y_test = split_xy_two_heads(datasets)

print(x_train_price.shape, x_train_temp.shape, y_train.shape)

(13240, 12, 1) (13240, 12, 1) (13240,)


In [29]:
def cnn_head(window_size: int) -> Model:
    inputs = Input((window_size, 1))

    x = Conv1D(64, 3, padding='same', activation='relu')(inputs)
    x = Conv1D(64, 3, padding='same', activation='relu')(x)
    x = MaxPool1D(2)(x)

    x = Conv1D(128, 3, padding='same', activation='relu')(x)
    x = Conv1D(128, 3, padding='same', activation='relu')(x)
    x = MaxPool1D(2)(x)

    x = Flatten()(x)

    outputs = Dense(128, activation='relu')(x)
    head = Model(inputs, outputs)

    return head

In [30]:
def get_cnn_two_heads(window_size: int) -> Model:
    
    price_inputs = Input((window_size, 1))
    temp_inputs = Input((window_size, 1))

    x_price = cnn_head(window_size)(price_inputs)
    x_temp = cnn_head(window_size)(temp_inputs)

    x = Concatenate()([x_price, x_temp])

    outputs = Dense(1, activation='linear')(x)

    cnn_model = Model([price_inputs, temp_inputs], outputs)

    cnn_model.compile(
        optimizer=Adam()
        , loss=mean_squared_error
        , metrics=['MAE']
    )
    return cnn_model

two_head_cnn = get_cnn_two_heads(window_size)
two_head_cnn_history = two_head_cnn.fit([x_train_price, x_train_temp], y_train, validation_data=([x_val_price, x_val_temp], y_val), epochs=25, batch_size=512)
#plot_model(model, expand_nested=True, show_layer_activations=True, show_shapes=True)

Epoch 1/25
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - MAE: 52.1013 - loss: 4333.9800 - val_MAE: 21.0024 - val_loss: 910.0402
Epoch 2/25
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - MAE: 17.4509 - loss: 571.8876 - val_MAE: 11.2513 - val_loss: 274.8569
Epoch 3/25
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - MAE: 10.1027 - loss: 215.5587 - val_MAE: 9.5697 - val_loss: 211.3830
Epoch 4/25
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - MAE: 9.0070 - loss: 171.8682 - val_MAE: 9.4837 - val_loss: 207.8503
Epoch 5/25
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - MAE: 8.7639 - loss: 166.0415 - val_MAE: 8.8423 - val_loss: 189.1170
Epoch 6/25
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - MAE: 8.3928 - loss: 152.6935 - val_MAE: 8.5648 - val_loss: 183.4358
Epoch 7/25
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - MAE: 8.1617 - loss: 147.3051 - val_MAE: 8.6622 - val_loss: 186.2657
Epoch 8/25
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - MAE: 7.9772 - loss: 137.4889 - val_MAE: 8.6503 - val_loss: 185.5400
Epoch 9/25
26/26 ━━━━━━━━━

In [31]:
y_pred_two_head_cnn = two_head_cnn.predict([x_test_price, x_test_temp])

plot_testset = datasets.get('test').with_columns(
    pl.Series('y_pred_two_head_cnn', y_pred_two_head_cnn.flatten()),
    ).select(pl.exclude('belpex_values', 'date'))
plot_testset.head(3)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step


temp,len,target,target_date,y_pred_two_head_cnn
list[f64],u32,f64,datetime[μs],f32
"[5.06, 4.96, … 6.32]",12,100.42,2025-02-28 12:00:00,103.779892
"[4.96, 5.03, … 7.3]",12,95.7,2025-02-28 13:00:00,94.582748
"[5.03, 4.95, … 8.23]",12,98.07,2025-02-28 14:00:00,92.583389


In [32]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=plot_testset['target_date'],
    y=plot_testset['target'],
    mode='lines',
    name='actual'
))

fig.add_trace(go.Scatter(
    x=plot_testset['target_date'],
    y=plot_testset['y_pred_two_head_cnn'],
    mode='lines',
    name='cnn_predicted'
))

fig.update_layout(
    title='Test Set Predictions',
    xaxis_title='Date',
    yaxis_title='Belpex Value',
    xaxis=dict(
        rangeslider=dict(visible=True),
        type='date'
    ),
    width=1200,
    height=600
)

fig.show()

# multistep prediction


Contrairement aux exemples précédents, nous allons prédire une séquence de valeurs.

<br>

In [33]:
def build_multisteps_dataset(data, window_size, dataset_name):
    step = window_size//2
    return data.filter(pl.col('dataset') == dataset_name)\
            .group_by_dynamic('date', every=f'{step}h', period=f'{window_size}h')\
            .agg(
                pl.col('belpex_values'),
                pl.col('date').alias('dates'),
                pl.len(),
                )\
            .with_columns(
                target = pl.col('belpex_values').list.slice(step).shift(-1),
                target_date = pl.col('dates').list.slice(step).shift(-1),
                )\
            .filter(
                (pl.col('len') == window_size) 
                &  ~pl.col('target_date').is_null()
                & (pl.col('target').list.len() == step)
                )
    
def split_xy(data: pl.DataFrame) -> tuple[np.array, np.array]:
    x = np.array(data.get_column('belpex_values').to_list())
    x = x[..., np.newaxis]
    y = np.array(data.get_column('target').to_list())
    return x, y

In [34]:
window_size = 12
data = split_dataset(window_size)

datasets = {}
for split in ['train', 'val', 'test']:
    datasets[split] = build_multisteps_dataset(data, window_size, split)

x_train, y_train = split_xy(datasets.get('train'))
x_val, y_val = split_xy(datasets.get('val'))
x_test, y_test = split_xy(datasets.get('test'))
print(x_train.shape)

(2490, 12, 1)


In [35]:
from keras.layers import LSTM, Bidirectional, BatchNormalization
from keras.optimizers.schedules import CosineDecay
from keras.optimizers import AdamW

def get_lstm(window_size: int) -> Model:
    inputs = Input((window_size, 1))
    x = LSTM(128, activation='relu', return_sequences=True)(inputs)
    x = LSTM(128, activation='relu', return_sequences=True)(x)
    x = LSTM(128, activation='relu')(x)
    outputs = Dense(window_size-(window_size//2), activation='linear')(x)

    lstm_model = Model(inputs, outputs)
    lstm_model.compile(
        optimizer=AdamW(),
        loss=mean_squared_error,
        metrics=['MAE']
    )

    return lstm_model

lstm_model = get_lstm(window_size)
lstm_history = lstm_model.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=50, batch_size=512)

Epoch 1/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - MAE: 87.0068 - loss: 10081.4141 - val_MAE: 65.3756 - val_loss: 6142.1919
Epoch 2/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - MAE: 46.3371 - loss: 3218.1614 - val_MAE: 30.5595 - val_loss: 1892.7605
Epoch 3/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - MAE: 30.6515 - loss: 1584.2108 - val_MAE: 30.0017 - val_loss: 1772.5736
Epoch 4/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - MAE: 28.4777 - loss: 1394.0094 - val_MAE: 28.7121 - val_loss: 1748.8234
Epoch 5/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - MAE: 26.6709 - loss: 1272.4846 - val_MAE: 28.2893 - val_loss: 1648.3667
Epoch 6/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - MAE: 25.9938 - loss: 1192.5546 - val_MAE: 26.9010 - val_loss: 1545.4812
Epoch 7/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - MAE: 24.5790 - loss: 1081.4911 - val_MAE: 26.3544 - val_loss: 1453.0197
Epoch 8/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - MAE: 24.2792 - loss: 1055.6322 - val_MAE: 26.0077 - val_loss: 1467.0544
Epoch 9/50
5/5 

In [36]:
y_pred = lstm_model.predict(x_test)
plot_testset = datasets.get('test')\
    .select(
        pl.col('target_date', 'target'),
        pl.Series('y_pred', y_pred, pl.List)
        )\
    .explode(('target_date', 'target', 'y_pred'))
plot_testset.head(3)


11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step


target_date,target,y_pred
datetime[μs],f64,f32
2025-02-28 12:00:00,100.42,108.835144
2025-02-28 13:00:00,95.7,97.067841
2025-02-28 14:00:00,98.07,96.181419


In [37]:
fig = go.Figure()


fig.add_trace(go.Scatter(
    x=plot_testset['target_date'],
    y=plot_testset['target'],
    mode='lines',
    name='actual'
))

fig.add_trace(go.Scatter(
    x=plot_testset['target_date'],
    y=plot_testset['y_pred'],
    mode='lines',
    name='lstm_predicted'
))

fig.update_layout(
    title='Test Set Predictions',
    xaxis_title='Date',
    yaxis_title='Belpex Value',
    xaxis=dict(
        rangeslider=dict(visible=True),
        type='date'
    ),
    width=1200,
    height=600
)

fig.show()